# Week 8: Build an Agentic AI Pipeline

**Name:** Sahil Yadav

**Task:** Single Agent Systems & Agent Pipelines

## Theory

An Agentic AI Pipeline is a system where an AI agent performs tasks by following a structured workflow. It uses a stateful directed graph where nodes represent tasks and edges define how data flows between them. The agent can route queries to different tools, retry on failures, and track its actions for evaluation.

## Imports

In [ ]:
import json
import time
import random

## Define Tools with JSON Schema

In [ ]:
TOOLS = {
    "calculator": {
        "name": "calculator",
        "description": "Performs basic math operations",
        "input_schema": {"expression": "string"},
        "output_schema": {"result": "number"}
    },
    "keyword_extractor": {
        "name": "keyword_extractor",
        "description": "Extracts keywords from text",
        "input_schema": {"text": "string"},
        "output_schema": {"keywords": "list"}
    },
    "general_responder": {
        "name": "general_responder",
        "description": "Handles general queries",
        "input_schema": {"query": "string"},
        "output_schema": {"response": "string"}
    }
}

print("Available Tools:")
for name, tool in TOOLS.items():
    print(f"  - {name}: {tool['description']}")

## Tool Functions

In [ ]:
def calculator_tool(expression):
    try:
        result = eval(expression)
        return {"result": result, "status": "success"}
    except Exception as e:
        return {"error": str(e), "status": "failed"}


def keyword_extractor_tool(text):
    stop_words = {"the", "is", "a", "an", "in", "of", "and", "to", "for", "it", "on", "that", "this", "with"}
    words = text.lower().split()
    keywords = [w for w in words if w not in stop_words and len(w) > 2]
    return {"keywords": list(set(keywords)), "status": "success"}


def general_responder_tool(query):
    return {"response": f"I received your query: '{query}'. This is a general response.", "status": "success"}


# quick test
print("Calculator test:", calculator_tool("25 + 17 * 3"))
print("Keyword test:", keyword_extractor_tool("Python is a great language for AI"))

## Conditional Routing

Routes the query to the right tool based on keywords in the input.

In [ ]:
def route_query(query):
    query_lower = query.lower()
    if any(word in query_lower for word in ["calculate", "math", "add", "subtract", "multiply", "+", "-", "*", "/"]):
        return "calculator"
    elif any(word in query_lower for word in ["keyword", "extract", "words", "find words"]):
        return "keyword_extractor"
    else:
        return "general_responder"


# test routing
test_queries = ["calculate 5+3", "extract keywords from this", "hello there"]
for q in test_queries:
    print(f"  '{q}' -> {route_query(q)}")

## Error Handling with Retry Logic

Uses try-except and a retry mechanism to handle failures. There is a simulated 20% failure rate to test retry.

In [ ]:
def execute_with_retry(tool_func, tool_input, max_retries=3):
    for attempt in range(1, max_retries + 1):
        try:
            if random.random() < 0.2:
                raise ConnectionError("Simulated network error")
            result = tool_func(tool_input)
            return result, attempt
        except Exception as e:
            print(f"      [Retry {attempt}/{max_retries}] Error: {e}")
            if attempt == max_retries:
                return {"error": str(e), "status": "failed"}, attempt
            time.sleep(0.3)

## Agent Pipeline

The main pipeline class. It maintains state across nodes, routes queries, executes tools with retry, records trajectory, and tracks metrics.

In [ ]:
class AgentPipeline:

    def __init__(self):
        self.state = {}
        self.trajectory = []
        self.metrics = {"total_tasks": 0, "completed": 0, "failed": 0, "total_retries": 0, "total_time": 0}

    def run(self, query):
        start_time = time.time()
        self.metrics["total_tasks"] += 1
        print(f"\nQuery: {query}")
        print("-" * 50)

        # node 1 - analyze query
        self.state["query"] = query
        print(f"  [Node 1] QueryAnalyzer - analyzing query")
        self.trajectory.append({"node": "QueryAnalyzer", "input": query})

        # node 2 - route to tool
        tool_name = route_query(query)
        self.state["tool"] = tool_name
        print(f"  [Node 2] Router - routed to: {tool_name}")
        self.trajectory.append({"node": "Router", "routed_to": tool_name})

        # node 3 - execute tool
        tool_map = {
            "calculator": calculator_tool,
            "keyword_extractor": keyword_extractor_tool,
            "general_responder": general_responder_tool
        }

        if tool_name == "calculator":
            tool_input = query.lower().replace("calculate", "").replace("math", "").strip()
        else:
            tool_input = query

        print(f"  [Node 3] ToolExecutor - running {tool_name}")
        result, attempts = execute_with_retry(tool_map[tool_name], tool_input)
        self.metrics["total_retries"] += (attempts - 1)
        self.trajectory.append({"node": "ToolExecutor", "tool": tool_name, "result": result, "attempts": attempts})
        self.state["result"] = result

        # node 4 - response
        if result.get("status") == "success":
            self.metrics["completed"] += 1
            print(f"  [Node 4] ResponseGenerator - success")
        else:
            self.metrics["failed"] += 1
            print(f"  [Node 4] ResponseGenerator - failed after {attempts} attempts")

        elapsed = time.time() - start_time
        self.metrics["total_time"] += elapsed
        self.trajectory.append({"node": "ResponseGenerator", "output": result, "time": round(elapsed, 3)})

        print(f"  Result: {json.dumps(result, indent=4)}")
        print(f"  Time: {elapsed:.3f}s | Attempts: {attempts}")
        return result

    def show_trajectory(self):
        print("\nTrajectory Evaluation")
        print("-" * 50)
        for i, step in enumerate(self.trajectory):
            print(f"  Step {i+1}: {json.dumps(step)}")

    def show_metrics(self):
        total = self.metrics["total_tasks"]
        completed = self.metrics["completed"]
        rate = (completed / total * 100) if total > 0 else 0
        print("\nMetrics Dashboard")
        print("-" * 50)
        print(f"  Total Tasks:       {total}")
        print(f"  Completed:         {completed}")
        print(f"  Failed:            {self.metrics['failed']}")
        print(f"  Completion Rate:   {rate:.1f}%")
        print(f"  Total Retries:     {self.metrics['total_retries']}")
        print(f"  Total Time:        {self.metrics['total_time']:.3f}s")
        if total > 0:
            print(f"  Avg Time/Task:     {self.metrics['total_time']/total:.3f}s")

## Running the Pipeline

In [ ]:
agent = AgentPipeline()

queries = [
    "calculate 25 + 17 * 3",
    "extract keywords from: The agent pipeline uses nodes and edges for routing",
    "What is the weather today?",
    "math 100 / 4 + 50",
    "find words in: Python is a great programming language for AI",
]

for q in queries:
    agent.run(q)

## Trajectory Evaluation

In [ ]:
agent.show_trajectory()

## Metrics

In [ ]:
agent.show_metrics()

---

## Quiz - Single Agent Systems & Agent Pipelines

**Q1. Explain the concept of a stateful directed graph in agent pipelines. How does it differ from a simple linear pipeline?**

A stateful directed graph is a workflow where each step can store and use information from previous steps. The workflow is made up of nodes and edges that define how data moves through the system. It can support branching, looping, and decision-making. This makes it more flexible and intelligent. In contrast, a linear pipeline follows a fixed sequence of steps from start to finish without remembering previous states or changing its path.

**Q2. Describe the role of nodes and edges in an agent workflow. Give an example of each.**

Nodes are the individual tasks or actions performed in an agent workflow. They can represent operations such as processing a query, calling a tool, or generating a response. Edges are the connections between nodes that determine how information flows. For example, a "Calculator Tool" can be a node, while the path connecting query analysis to the calculator is an edge. Together, nodes and edges define the complete workflow.

**Q3. What is conditional routing in an agent system? Design a simple rule-based routing logic for three different query types.**

Conditional routing is the process of directing a query to different tools or actions based on its intent. It helps the agent choose the most appropriate response method. For example, if a query contains the word "calculate", it is sent to the Calculator Tool. If it contains "keywords", it is sent to the Keyword Extraction Tool. All other queries can be handled by a General Response module.

**Q4. Why are cycles (loops) important in agent pipelines? Provide a use case where a retry loop is necessary.**

Cycles or loops allow an agent to repeat a process until a desired result is achieved. They are useful when tasks may fail or require multiple attempts. For example, if an API request fails due to a temporary network issue, the agent can retry the request instead of immediately returning an error. This improves reliability and robustness. Without loops, the workflow would stop after a single failure.

**Q5. Explain how a single-agent system can simulate multi-agent behavior internally.**

A single-agent system can simulate multiple agents by dividing its work into different roles. It may first analyze the query, then decide which tool to use, and finally generate a response. Each role behaves like a separate agent even though they are all executed by one system. This approach reduces complexity while maintaining flexibility. As a result, a single agent can perform tasks similar to a multi-agent architecture.

**Q6. What are JSON schema tools? How do they help in structuring tool inputs and outputs?**

JSON schema tools define a standard structure for data exchanged between agents and tools. They specify required fields, data types, and expected formats. This ensures that inputs are valid before processing begins. It also makes outputs consistent and easier to interpret. Using JSON schemas reduces errors and improves communication between different components of an agent system.

**Q7. Compare sequential tool calls and parallel tool calls. When would you prefer one over the other?**

Sequential tool calls are executed one after another, where each step depends on the result of the previous step. Parallel tool calls execute multiple independent tasks at the same time. Sequential execution is preferred when tasks have dependencies. Parallel execution is useful when tasks are unrelated and can be completed simultaneously. Using parallel calls can significantly reduce response time and improve efficiency.

**Q8. How would you implement error handling in a tool-using agent? Provide at least two strategies.**

Error handling helps an agent continue operating even when problems occur. One strategy is using try-except blocks to catch exceptions and return meaningful error messages. Another strategy is implementing retry mechanisms that automatically repeat failed operations. Logging errors is also useful for debugging and monitoring. These techniques improve reliability and help maintain a better user experience.

**Q9. What is trajectory evaluation in agent systems? Why is it important beyond just checking final output?**

Trajectory evaluation examines the entire sequence of actions taken by an agent to solve a task. It focuses on the decisions, tool calls, and intermediate steps used during execution. This helps identify mistakes that may not be visible in the final answer. It is useful for debugging, optimization, and improving agent performance. By evaluating the full trajectory, developers gain a deeper understanding of agent behavior.

**Q10. Define task completion rate and cost metrics. How would you measure and optimize them in a real-world system?**

Task completion rate measures the percentage of tasks successfully completed by an agent. Cost metrics represent the resources consumed, such as API calls, processing time, or monetary expenses. These metrics can be measured by tracking agent performance over multiple tasks. To optimize them, developers can improve routing accuracy, reduce unnecessary tool calls, and use efficient algorithms. Balancing high completion rates with low costs is important for real-world deployment.